# Parisian Options — Monte Carlo Pricing under GBM

A **Parisian barrier** option is knocked in or out not when the underlying first *touches* a barrier,
but when it *spends a consecutive time window $D$* past the barrier.  This makes them harder to
trigger than standard barriers, so Parisian knockouts are always worth **at least as much** as
the corresponding standard barrier option.

Two flavours are implemented:

| Flavour | Clock behaviour | Condition fires when... |
|---------|----------------|-------------------------|
| **Standard (resetting)** | Resets each time path re-crosses barrier | Any *single* excursion exceeds $D$ |
| **Cumulative** | Accumulates total time past barrier | *Total* time past barrier exceeds $D$ |

**Reference**: Chesney, Jeanblanc-Picqué & Yor (1997), *Brownian Excursions and Parisian Barrier Options*.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from foureng.mc.parisian_mc import parisian_mc_price
from foureng.mc.paths import gbm_paths
from foureng.products.parisian import ParisianOption
from foureng.analytics.bsm_barrier import bsm_barrier_price, bsm_call, bsm_put

# Shared parameters
S0, K, H = 100.0, 100.0, 90.0
r, q, sigma, T = 0.05, 0.0, 0.20, 1.0
D = 0.1  # Parisian window: 0.1 years = ~25 trading days

MC_PATHS = 100_000
N_STEPS  = 800
SEED     = 42

## 1. Visualising Parisian excursions
We plot a few GBM sample paths and shade regions where the path is below the barrier.
The Parisian condition fires if any shaded region runs for longer than $D$.

In [ ]:
rng = np.random.default_rng(SEED)
n_plot, n_steps_plot = 5, 500
paths_plot = gbm_paths(S0, r, q, T, sigma, n_plot, n_steps_plot, rng)
t_grid = np.linspace(0, T, n_steps_plot + 1)

fig, ax = plt.subplots(figsize=(10, 4))
colors = plt.cm.tab10.colors
for i in range(n_plot):
    path = paths_plot[i]
    ax.plot(t_grid, path, color=colors[i], lw=1.2, alpha=0.85, label=f'Path {i+1}')
    # Shade regions below H
    below = path < H
    ax.fill_between(t_grid, H, path, where=below, color=colors[i], alpha=0.15)

ax.axhline(H, color='black', lw=1.5, ls='--', label=f'Barrier H={H}')
ax.axhline(K, color='grey', lw=1.0, ls=':', label=f'Strike K={K}')
ax.set_xlabel('Time (years)')
ax.set_ylabel('Spot price')
ax.set_title(f'GBM paths — shaded regions = time below barrier (Parisian window D={D}y)')
ax.legend(loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

## 2. In-out parity verification
The key parity: **knockout + knockin = vanilla**.
This holds for both standard and cumulative Parisian types.

In [ ]:
vanilla = bsm_call(S0, K, r, q, T, sigma)
print(f'BSM vanilla call: {vanilla:.4f}')
print()

for ptype in ('standard', 'cumulative'):
    ko, se_ko = parisian_mc_price(
        S0, K, H, r, q, T, sigma, D,
        cp=1, direction='down', knockout=True, parisian_type=ptype,
        n_paths=MC_PATHS, n_steps=N_STEPS, seed=SEED, antithetic=True,
    )
    ki, se_ki = parisian_mc_price(
        S0, K, H, r, q, T, sigma, D,
        cp=1, direction='down', knockout=False, parisian_type=ptype,
        n_paths=MC_PATHS, n_steps=N_STEPS, seed=SEED+1, antithetic=True,
    )
    print(f'{ptype.capitalize()} Parisian:')
    print(f'  Knockout = {ko:.4f} ± {2*se_ko:.4f}')
    print(f'  Knockin  = {ki:.4f} ± {2*se_ki:.4f}')
    print(f'  KO + KI  = {ko+ki:.4f}  (vanilla = {vanilla:.4f})  diff = {abs(ko+ki-vanilla):.5f}')
    print()

## 3. Parisian window $D$ sweep
Vary the excursion window $D$ and compare:
- Standard Parisian knockout call
- Standard barrier knockout call (reference)
- Vanilla BSM call

In [ ]:
D_grid = [0.01, 0.02, 0.05, 0.10, 0.20, 0.30, 0.50, 0.70, 0.90]
par_prices, par_errors = [], []

for D_val in D_grid:
    p, se = parisian_mc_price(
        S0, K, H, r, q, T, sigma, D_val,
        cp=1, direction='down', knockout=True, parisian_type='standard',
        n_paths=MC_PATHS, n_steps=N_STEPS, seed=SEED, antithetic=True,
    )
    par_prices.append(p)
    par_errors.append(2 * se)  # 2-sigma band

barrier_price = bsm_barrier_price(S0, K, H, r, q, T, sigma, 'down_out', cp=1)
vanilla_price = bsm_call(S0, K, r, q, T, sigma)

fig, ax = plt.subplots(figsize=(8, 4))
ax.errorbar(D_grid, par_prices, yerr=par_errors, fmt='o-', label='Parisian KO (standard)', capsize=3)
ax.axhline(barrier_price, ls='--', color='tomato', label=f'Standard barrier KO = {barrier_price:.4f}')
ax.axhline(vanilla_price, ls=':', color='steelblue', label=f'Vanilla call = {vanilla_price:.4f}')
ax.set_xlabel('Excursion window D (years)')
ax.set_ylabel('Option price')
ax.set_title('Parisian KO Call vs excursion window D')
ax.legend()
ax.set_xscale('log')
plt.tight_layout()
plt.show()
print('As D→0, Parisian→standard barrier. As D→T, Parisian→vanilla.')

## 4. Standard vs Cumulative comparison
For the same $D$, the cumulative Parisian fires more easily (total time, not consecutive),
so its knockout option is worth less.

In [ ]:
std_prices, cum_prices = [], []

for D_val in D_grid:
    p_std, _ = parisian_mc_price(
        S0, K, H, r, q, T, sigma, D_val,
        cp=1, direction='down', knockout=True, parisian_type='standard',
        n_paths=MC_PATHS, n_steps=N_STEPS, seed=SEED, antithetic=True,
    )
    p_cum, _ = parisian_mc_price(
        S0, K, H, r, q, T, sigma, D_val,
        cp=1, direction='down', knockout=True, parisian_type='cumulative',
        n_paths=MC_PATHS, n_steps=N_STEPS, seed=SEED, antithetic=True,
    )
    std_prices.append(p_std)
    cum_prices.append(p_cum)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(D_grid, std_prices, 'o-', label='Standard (resetting)')
ax.plot(D_grid, cum_prices, 's--', label='Cumulative')
ax.axhline(barrier_price, ls=':', color='tomato', label='Standard barrier KO')
ax.set_xlabel('Excursion window D (years)')
ax.set_ylabel('Knockout call price')
ax.set_title('Standard vs Cumulative Parisian KO Call')
ax.legend()
ax.set_xscale('log')
plt.tight_layout()
plt.show()

## 5. Strike sweep — Parisian smiles
Plot Parisian KO call prices across strikes for a fixed window $D=0.1$.

In [ ]:
strikes = np.array([80, 85, 90, 95, 100, 105, 110, 115, 120], dtype=float)
D_fixed = 0.1

par_std, par_cum, vanilla_arr, barrier_arr = [], [], [], []
for K_val in strikes:
    p_std, _ = parisian_mc_price(
        S0, K_val, H, r, q, T, sigma, D_fixed,
        cp=1, direction='down', knockout=True, parisian_type='standard',
        n_paths=60_000, n_steps=N_STEPS, seed=SEED, antithetic=True,
    )
    p_cum, _ = parisian_mc_price(
        S0, K_val, H, r, q, T, sigma, D_fixed,
        cp=1, direction='down', knockout=True, parisian_type='cumulative',
        n_paths=60_000, n_steps=N_STEPS, seed=SEED, antithetic=True,
    )
    par_std.append(p_std)
    par_cum.append(p_cum)
    vanilla_arr.append(bsm_call(S0, K_val, r, q, T, sigma))
    barrier_arr.append(bsm_barrier_price(S0, K_val, H, r, q, T, sigma, 'down_out', cp=1))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(strikes, vanilla_arr, 'k--', label='Vanilla BSM call', lw=1.5)
ax.plot(strikes, barrier_arr, 'r:', label='Standard barrier KO', lw=1.5)
ax.plot(strikes, par_std, 'bo-', label=f'Parisian KO (standard, D={D_fixed}y)')
ax.plot(strikes, par_cum, 'gs--', label=f'Parisian KO (cumulative, D={D_fixed}y)')
ax.set_xlabel('Strike')
ax.set_ylabel('Call price')
ax.set_title(f'Parisian KO Call across strikes  (S₀={S0}, H={H}, D={D_fixed}y)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 6. Convergence in number of time steps
Discrete monitoring underestimates the continuous-barrier Parisian price.
We show convergence as `n_steps` increases.

In [ ]:
steps_grid = [50, 100, 200, 400, 800, 1600]
conv_prices, conv_errors = [], []

for n_st in steps_grid:
    p, se = parisian_mc_price(
        S0, K, H, r, q, T, sigma, D,
        cp=1, direction='down', knockout=True, parisian_type='standard',
        n_paths=100_000, n_steps=n_st, seed=SEED, antithetic=True,
    )
    conv_prices.append(p)
    conv_errors.append(2 * se)

fig, ax = plt.subplots(figsize=(8, 4))
ax.errorbar(steps_grid, conv_prices, yerr=conv_errors, fmt='o-', capsize=3)
ax.set_xlabel('Number of time steps')
ax.set_ylabel('Parisian KO call price')
ax.set_title('Convergence in n_steps (D=0.1y, standard Parisian)')
ax.set_xscale('log')
plt.tight_layout()
plt.show()

## 7. ParisianOption product dataclass

In [ ]:
from foureng.mc.parisian_mc import parisian_mc_price_from_product

p = ParisianOption(
    strike=100.0,
    barrier=90.0,
    maturity=1.0,
    excursion_window=0.1,
    cp=1,
    direction='down',
    knockout=True,
    parisian_type='standard',
)
print('Product spec:', p)

price, se = parisian_mc_price_from_product(
    p, S0=100.0, r=0.05, q=0.0, sigma=0.20,
    n_paths=100_000, n_steps=800, seed=42,
)
print(f'\nParisian KO call price: {price:.4f} ± {2*se:.4f} (2σ)')